#Sarah Cheulkar- BE CMPN C- 23102C0060
Assignment 3 — R Programming
## Multi-Source Retail Sales Data Integration and Analysis

**Platform:** R / Google Colab  
**Domain:** Retail Analytics  
**Dataset:** UCI Online Retail Dataset

**Workflow:** CSV + JSON + Excel → Import & Clean → Integrate → Analyze → SQLite → Business Insights


## Task 0 — Install and Load Required Packages
Packages used: `readr`, `jsonlite`, `readxl`, `openxlsx`, `dplyr`, `DBI`, and `RSQLite`.


In [1]:
options(repos = c(CRAN = "https://cloud.r-project.org"))

packages <- c("readr", "jsonlite", "readxl", "openxlsx", "dplyr", "DBI", "RSQLite")

for (pkg in packages) {
  if (!requireNamespace(pkg, quietly = TRUE)) install.packages(pkg, quiet = TRUE)
}

library(readr)
library(jsonlite)
library(readxl)
library(openxlsx)
library(dplyr)
library(DBI)
library(RSQLite)



Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




## Dataset Preparation
The assignment requires `transactions.csv`, `products.json`, and `customers.xlsx`. The UCI Online Retail dataset is downloaded and separated into these three required source formats so the complete heterogeneous-data workflow is reproducible in Colab.


In [2]:
# Download UCI Online Retail dataset
uci_url <- "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"
download.file(uci_url, "Online_Retail.xlsx", mode = "wb")

online_retail <- readxl::read_excel("Online_Retail.xlsx")
head(online_retail)


InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
<chr>,<chr>,<chr>,<dbl>,<dttm>,<dbl>,<dbl>,<chr>
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850,United Kingdom


In [3]:
# Inspect original dataset
str(online_retail)
dim(online_retail)
summary(online_retail)


tibble [541,909 × 8] (S3: tbl_df/tbl/data.frame)
 $ InvoiceNo  : chr [1:541909] "536365" "536365" "536365" "536365" ...
 $ StockCode  : chr [1:541909] "85123A" "71053" "84406B" "84029G" ...
 $ Description: chr [1:541909] "WHITE HANGING HEART T-LIGHT HOLDER" "WHITE METAL LANTERN" "CREAM CUPID HEARTS COAT HANGER" "KNITTED UNION FLAG HOT WATER BOTTLE" ...
 $ Quantity   : num [1:541909] 6 6 8 6 6 2 6 6 6 32 ...
 $ InvoiceDate: POSIXct[1:541909], format: "2010-12-01 08:26:00" "2010-12-01 08:26:00" ...
 $ UnitPrice  : num [1:541909] 2.55 3.39 2.75 3.39 3.39 7.65 4.25 1.85 1.85 1.69 ...
 $ CustomerID : num [1:541909] 17850 17850 17850 17850 17850 ...
 $ Country    : chr [1:541909] "United Kingdom" "United Kingdom" "United Kingdom" "United Kingdom" ...


[1] 541909      8

     InvoiceNo          StockCode         Description        Quantity         
 Length   :541909   Length   :541909   Length   :541909   Min.   :-80995.000  
 N.unique : 25900   N.unique :  4070   N.unique :  4211   1st Qu.:     1.000  
 N.blank  :     0   N.blank  :     0   N.blank  :     0   Median :     3.000  
 Min.nchar:     6   Min.nchar:     1   Min.nchar:     1   Mean   :     9.552  
 Max.nchar:     7   Max.nchar:    12   Max.nchar:    35   3rd Qu.:    10.000  
                                       NAs      :  1454   Max.   : 80995.000  
                                                                              
  InvoiceDate                    UnitPrice            CustomerID    
 Min.   :2010-12-01 08:26:00   Min.   :-11062.060   Min.   :12346   
 1st Qu.:2011-03-28 11:34:00   1st Qu.:     1.250   1st Qu.:13953   
 Median :2011-07-19 17:17:00   Median :     2.080   Median :15152   
 Mean   :2011-07-04 13:34:57   Mean   :     4.611   Mean   :15288   
 3rd Qu.:2011-10-19 11:

In [4]:
# Create the required CSV source
transactions_source <- online_retail %>%
  transmute(
    InvoiceNo = as.character(InvoiceNo),
    StockCode = as.character(StockCode),
    CustomerID = as.character(CustomerID),
    Quantity = as.numeric(Quantity),
    InvoiceDate = as.character(InvoiceDate)
  )

write_csv(transactions_source, "transactions.csv")

# Create the required JSON source
products_source <- online_retail %>%
  transmute(
    StockCode = as.character(StockCode),
    Description = as.character(Description),
    UnitPrice = as.numeric(UnitPrice)
  ) %>%
  distinct(StockCode, .keep_all = TRUE)

write_json(products_source, "products.json", pretty = TRUE, na = "null")

# Create the required Excel source
customers_source <- online_retail %>%
  transmute(
    CustomerID = as.character(CustomerID),
    Country = as.character(Country)
  ) %>%
  filter(!is.na(CustomerID)) %>%
  distinct(CustomerID, .keep_all = TRUE)

write.xlsx(customers_source, "customers.xlsx", overwrite = TRUE)

list.files(pattern = "^(transactions|products|customers)\\.")


[1] "customers.xlsx"   "products.json"    "transactions.csv"

# Task 1 — Import and Clean the Data

In [5]:
# Import CSV
transactions <- read_csv("transactions.csv", show_col_types = FALSE)

# Import JSON
products <- as.data.frame(fromJSON("products.json"), stringsAsFactors = FALSE)

# Import Excel
customers <- read_excel("customers.xlsx")

head(transactions)
head(products)
head(customers)


InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate
<chr>,<chr>,<dbl>,<dbl>,<dttm>
536365,85123A,17850,6,2010-12-01 08:26:00
536365,71053,17850,6,2010-12-01 08:26:00
536365,84406B,17850,8,2010-12-01 08:26:00
536365,84029G,17850,6,2010-12-01 08:26:00
536365,84029E,17850,6,2010-12-01 08:26:00
536365,22752,17850,2,2010-12-01 08:26:00


,StockCode,Description,UnitPrice
,<chr>,<chr>,<dbl>
1,85123A,WHITE HANGING HEART T-LIGHT HOLDER,2.55
2,71053,WHITE METAL LANTERN,3.39
3,84406B,CREAM CUPID HEARTS COAT HANGER,2.75
4,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,3.39
5,84029E,RED WOOLLY HOTTIE WHITE HEART.,3.39
6,22752,SET 7 BABUSHKA NESTING BOXES,7.65


CustomerID,Country
<chr>,<chr>
17850,United Kingdom
13047,United Kingdom
12583,France
13748,United Kingdom
15100,United Kingdom
15291,United Kingdom


In [6]:
# Inspect dimensions and structure
cat("Transactions:", nrow(transactions), "rows,", ncol(transactions), "columns\n")
cat("Products:", nrow(products), "rows,", ncol(products), "columns\n")
cat("Customers:", nrow(customers), "rows,", ncol(customers), "columns\n")

str(transactions)
str(products)
str(customers)


Transactions: 541909 rows, 5 columns
Products: 4070 rows, 3 columns
Customers: 4372 rows, 2 columns
spc_tbl_ [541,909 × 5] (S3: spec_tbl_df/tbl_df/tbl/data.frame)
 $ InvoiceNo  : chr [1:541909] "536365" "536365" "536365" "536365" ...
 $ StockCode  : chr [1:541909] "85123A" "71053" "84406B" "84029G" ...
 $ CustomerID : num [1:541909] 17850 17850 17850 17850 17850 ...
 $ Quantity   : num [1:541909] 6 6 8 6 6 2 6 6 6 32 ...
 $ InvoiceDate: POSIXct[1:541909], format: "2010-12-01 08:26:00" "2010-12-01 08:26:00" ...
 - attr(*, "spec")=
  .. cols(
  ..   InvoiceNo = col_character(),
  ..   StockCode = col_character(),
  ..   CustomerID = col_double(),
  ..   Quantity = col_double(),
  ..   InvoiceDate = col_datetime(format = "")
  .. )
 - attr(*, "problems")=<pointer: 0x5b8669a53f90> 
'data.frame':	4070 obs. of  3 variables:
 $ StockCode  : chr  "85123A" "71053" "84406B" "84029G" ...
 $ Description: chr  "WHITE HANGING HEART T-LIGHT HOLDER" "WHITE METAL LANTERN" "CREAM CUPID HEARTS COAT HANGE

### Cleaning decisions
- Remove duplicate records.
- Remove transactions with missing keys or non-positive quantities.
- Remove product records with missing keys/descriptions or non-positive prices.
- Remove customer records with missing IDs/country.
- Calculate `Revenue = Quantity × UnitPrice` after product integration because UnitPrice is stored in the product source.


In [7]:
# Standardize data types
transactions <- transactions %>%
  mutate(
    InvoiceNo = as.character(InvoiceNo),
    StockCode = as.character(StockCode),
    CustomerID = as.character(CustomerID),
    Quantity = as.numeric(Quantity),
    InvoiceDate = as.character(InvoiceDate)
  )

products <- products %>%
  mutate(
    StockCode = as.character(StockCode),
    Description = as.character(Description),
    UnitPrice = as.numeric(UnitPrice)
  )

customers <- customers %>%
  mutate(
    CustomerID = as.character(CustomerID),
    Country = as.character(Country)
  )


In [8]:
# Missing-value inspection
cat("Missing values - transactions:\n")
print(colSums(is.na(transactions)))

cat("\nMissing values - products:\n")
print(colSums(is.na(products)))

cat("\nMissing values - customers:\n")
print(colSums(is.na(customers)))


Missing values - transactions:
  InvoiceNo   StockCode  CustomerID    Quantity InvoiceDate 
          0           0      135080           0           0 

Missing values - products:
  StockCode Description   UnitPrice 
          0         176           0 

Missing values - customers:
CustomerID    Country 
         0          0 


In [9]:
# Duplicate inspection
cat("Duplicate transaction rows:", sum(duplicated(transactions)), "\n")
cat("Duplicate product rows:", sum(duplicated(products)), "\n")
cat("Duplicate customer rows:", sum(duplicated(customers)), "\n")


Duplicate transaction rows: 5429 
Duplicate product rows: 0 
Duplicate customer rows: 0 


In [10]:
# Clean transactions
transactions_clean <- transactions %>%
  distinct() %>%
  filter(
    !is.na(InvoiceNo),
    !is.na(StockCode),
    !is.na(CustomerID),
    !is.na(Quantity),
    Quantity > 0
  )

# Clean products
products_clean <- products %>%
  filter(
    !is.na(StockCode),
    !is.na(Description),
    !is.na(UnitPrice),
    UnitPrice > 0
  ) %>%
  distinct(StockCode, .keep_all = TRUE)

# Clean customers
customers_clean <- customers %>%
  filter(
    !is.na(CustomerID),
    !is.na(Country)
  ) %>%
  distinct(CustomerID, .keep_all = TRUE)

cat("Clean transactions:", nrow(transactions_clean), "\n")
cat("Clean products:", nrow(products_clean), "\n")
cat("Clean customers:", nrow(customers_clean), "\n")


Clean transactions: 392708 
Clean products: 3855 
Clean customers: 4372 


# Task 2 — Integrate the Multiple Data Sources
`left_join()` is used because transactions are the primary business records. It preserves transaction rows while attaching product and customer attributes. `anti_join()` identifies unmatched records.


In [11]:
# Identify unmatched product and customer keys
unmatched_products <- transactions_clean %>%
  anti_join(products_clean, by = "StockCode")

unmatched_customers <- transactions_clean %>%
  anti_join(customers_clean, by = "CustomerID")

cat("Unmatched product records:", nrow(unmatched_products), "\n")
cat("Unmatched customer records:", nrow(unmatched_customers), "\n")


Unmatched product records: 4831 
Unmatched customer records: 0 


In [12]:
# Integrate transaction, product and customer data
retail_sales <- transactions_clean %>%
  left_join(products_clean, by = "StockCode") %>%
  left_join(customers_clean, by = "CustomerID") %>%
  mutate(Revenue = Quantity * UnitPrice)

head(retail_sales)


InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate,Description,UnitPrice,Country,Revenue
<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<chr>,<dbl>
536365,85123A,17850,6,2010-12-01 08:26:00,WHITE HANGING HEART T-LIGHT HOLDER,2.55,United Kingdom,15.30
536365,71053,17850,6,2010-12-01 08:26:00,WHITE METAL LANTERN,3.39,United Kingdom,20.34
536365,84406B,17850,8,2010-12-01 08:26:00,CREAM CUPID HEARTS COAT HANGER,2.75,United Kingdom,22.00
536365,84029G,17850,6,2010-12-01 08:26:00,KNITTED UNION FLAG HOT WATER BOTTLE,3.39,United Kingdom,20.34
536365,84029E,17850,6,2010-12-01 08:26:00,RED WOOLLY HOTTIE WHITE HEART.,3.39,United Kingdom,20.34
536365,22752,17850,2,2010-12-01 08:26:00,SET 7 BABUSHKA NESTING BOXES,7.65,United Kingdom,15.30


In [13]:
# Verify final dimensions
cat("Final dataset:", nrow(retail_sales), "rows and", ncol(retail_sales), "columns\n")
dim(retail_sales)

# Identify rows with unmatched attributes
cat("Rows with missing product information:", sum(is.na(retail_sales$Description)), "\n")
cat("Rows with missing customer country:", sum(is.na(retail_sales$Country)), "\n")


Final dataset: 392708 rows and 9 columns


[1] 392708      9

Rows with missing product information: 4831 
Rows with missing customer country: 0 


# Task 3 — Sales and Customer Analysis

In [14]:
# 1. Total sales revenue
total_revenue <- sum(retail_sales$Revenue, na.rm = TRUE)
cat("Total Sales Revenue:", total_revenue, "\n")


Total Sales Revenue: 10752840 


In [15]:
# 2. Top 5 products based on revenue
top_5_products <- retail_sales %>%
  group_by(StockCode, Description) %>%
  summarise(Total_Revenue = sum(Revenue, na.rm = TRUE), .groups = "drop") %>%
  arrange(desc(Total_Revenue)) %>%
  slice_head(n = 5)

top_5_products


StockCode,Description,Total_Revenue
<chr>,<chr>,<dbl>
23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60
47566,PARTY BUNTING,142437.56
22423,REGENCY CAKESTAND 3 TIER,135604.80
85123A,WHITE HANGING HEART T-LIGHT HOLDER,93745.65
23166,MEDIUM CERAMIC TOP STORAGE JAR,81032.64


In [16]:
# 3. Top 5 countries based on revenue
top_5_countries <- retail_sales %>%
  group_by(Country) %>%
  summarise(Total_Revenue = sum(Revenue, na.rm = TRUE), .groups = "drop") %>%
  arrange(desc(Total_Revenue)) %>%
  slice_head(n = 5)

top_5_countries


Country,Total_Revenue
<chr>,<dbl>
United Kingdom,8861857.1
Netherlands,363884.5
EIRE,331660.2
Germany,263819.0
France,226975.6


In [17]:
# 4. Top 5 customers based on total purchase value
customer_value <- retail_sales %>%
  group_by(CustomerID) %>%
  summarise(Total_Purchase_Value = sum(Revenue, na.rm = TRUE), .groups = "drop") %>%
  arrange(desc(Total_Purchase_Value))

top_5_customers <- customer_value %>%
  slice_head(n = 5)

top_5_customers


CustomerID,Total_Purchase_Value
<chr>,<dbl>
18102,408760.0
14646,357531.1
17450,186038.0
14911,182690.5
16446,168472.5


### Customer value classification
Thresholds are based on quartiles of customer purchase values:
- Low Value: ≤ 25th percentile
- Medium Value: >25th and ≤50th percentile
- High Value: >50th and ≤75th percentile
- Premium: >75th percentile


In [18]:
q25 <- quantile(customer_value$Total_Purchase_Value, 0.25, na.rm = TRUE)
q50 <- quantile(customer_value$Total_Purchase_Value, 0.50, na.rm = TRUE)
q75 <- quantile(customer_value$Total_Purchase_Value, 0.75, na.rm = TRUE)

thresholds <- data.frame(
  Level = c("25th percentile", "50th percentile", "75th percentile"),
  Threshold = c(q25, q50, q75)
)

thresholds


,Level,Threshold
,<chr>,<dbl>
25%,25th percentile,361.005
50%,50th percentile,802.600
75%,75th percentile,2000.640


In [19]:
# Classify customers using case_when()
customer_value <- customer_value %>%
  mutate(
    Value_Class = case_when(
      Total_Purchase_Value <= q25 ~ "Low Value",
      Total_Purchase_Value <= q50 ~ "Medium Value",
      Total_Purchase_Value <= q75 ~ "High Value",
      TRUE ~ "Premium"
    )
  )

head(customer_value)


CustomerID,Total_Purchase_Value,Value_Class
<chr>,<dbl>,<chr>
18102,408760.0,Premium
14646,357531.1,Premium
17450,186038.0,Premium
14911,182690.5,Premium
16446,168472.5,Premium
12415,156180.4,Premium


In [20]:
# Count customers by value class
customer_value %>%
  count(Value_Class)


Value_Class,n
<chr>,<int>
High Value,1084
Low Value,1085
Medium Value,1085
Premium,1085


### High-performing and underperforming markets
Revenue is used as the primary market-performance measure.


In [21]:
country_performance <- retail_sales %>%
  group_by(Country) %>%
  summarise(
    Total_Revenue = sum(Revenue, na.rm = TRUE),
    Transactions = n(),
    .groups = "drop"
  ) %>%
  arrange(desc(Total_Revenue))

high_performing_market <- country_performance %>% slice_head(n = 1)
underperforming_market <- country_performance %>% arrange(Total_Revenue) %>% slice_head(n = 1)

cat("High-performing market:\n")
print(high_performing_market)

cat("\nUnderperforming market:\n")
print(underperforming_market)


High-performing market:
# A tibble: 1 × 3
  Country        Total_Revenue Transactions
  <chr>                  <dbl>        <int>
1 United Kingdom      8861857.       349212

Underperforming market:
# A tibble: 1 × 3
  Country      Total_Revenue Transactions
  <chr>                <dbl>        <int>
1 Saudi Arabia          181.            9


### Market interpretation
The high-performing market is the country with the greatest total revenue. The underperforming market is the represented country with the lowest total revenue. These results provide a direct revenue-based comparison for management decision-making.


# Task 4 — SQLite Database and SQL Queries

In [22]:
# Create SQLite database
db_path <- "retail_sales.sqlite"

if (file.exists(db_path)) file.remove(db_path)

con <- dbConnect(SQLite(), db_path)

# Write integrated dataset to table retail_sales
dbWriteTable(
  con,
  "retail_sales",
  retail_sales,
  overwrite = TRUE
)

dbListTables(con)


[1] "retail_sales"

### SQL Query 1 — Top 5 customers based on revenue

In [23]:
top_customers_sql <- dbGetQuery(
  con,
  "SELECT CustomerID,
          SUM(Revenue) AS Total_Revenue
   FROM retail_sales
   GROUP BY CustomerID
   ORDER BY Total_Revenue DESC
   LIMIT 5"
)

top_customers_sql


CustomerID,Total_Revenue
<chr>,<dbl>
18102,408760.0
14646,357531.1
17450,186038.0
14911,182690.5
16446,168472.5


### SQL Query 2 — Total revenue by country

In [24]:
country_revenue_sql <- dbGetQuery(
  con,
  "SELECT Country,
          SUM(Revenue) AS Total_Revenue
   FROM retail_sales
   GROUP BY Country
   ORDER BY Total_Revenue DESC"
)

head(country_revenue_sql, 10)


,Country,Total_Revenue
,<chr>,<dbl>
1,United Kingdom,8861857.13
2,Netherlands,363884.48
3,EIRE,331660.17
4,Germany,263818.97
5,France,226975.60
6,Australia,173918.61
7,Spain,67426.09
8,Switzerland,66619.97
9,Japan,48600.22


In [25]:
# Verify SQLite table
dbGetQuery(con, "SELECT COUNT(*) AS Row_Count FROM retail_sales")


Row_Count
<int>
392708


In [26]:
# Close database connection
dbDisconnect(con)

cat("SQLite database created:", file.exists("retail_sales.sqlite"), "\n")


SQLite database created: TRUE 


# Three Business Insights

1. **Revenue concentration:** The top products and countries identify the largest contributors to sales revenue and therefore the areas where management can prioritize inventory and marketing.

2. **Customer value segmentation:** The Low Value, Medium Value, High Value and Premium groups provide a data-driven basis for differentiated customer retention and promotional strategies.

3. **Market prioritization:** Comparing country-level revenue identifies the strongest and weakest represented markets, helping management focus expansion on high-performing markets and investigate low-performing markets.


# Final Deliverables Checklist

- [x] CSV, JSON and Excel import
- [x] Missing-value handling
- [x] Duplicate handling
- [x] Invalid/zero quantity handling
- [x] Invalid/zero unit-price handling
- [x] Revenue attribute
- [x] `left_join()` integration
- [x] Unmatched-record identification
- [x] Final dimensions
- [x] Total revenue
- [x] Top 5 products
- [x] Top 5 countries
- [x] Top 5 customers
- [x] `case_when()` customer classification
- [x] High-performing and underperforming market
- [x] SQLite database and `retail_sales` table
- [x] Two SQL queries
- [x] Three business insights

### Files generated
- `transactions.csv`
- `products.json`
- `customers.xlsx`
- `retail_sales.sqlite`

**Submission:** Run all cells from top to bottom so code, output, comments and interpretations are visible. Export the completed notebook as PDF, retain the `.ipynb` and SQLite database, and upload the complete work to Git as required.
